In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/obesity_CVD_risk/train.csv')

# Display the first few rows of the dataset
print(train_data.head())

# Get a summary of the dataset
print(train_data.info())

# Check for missing values
print(train_data.isnull().sum())

# Visualize missing values
plt.figure(figsize=(10, 6))
sns.heatmap(train_data.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

# Distinguish column types
numeric_cols = train_data.select_dtypes(include=[np.number]).columns
categorical_cols = train_data.select_dtypes(include=['object', 'category']).columns

print("Numeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

# Descriptive statistics for numeric columns
print(train_data[numeric_cols].describe())

# Descriptive statistics for categorical columns
print(train_data[categorical_cols].describe())

# Check for anomalies in numeric columns
for col in numeric_cols:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=train_data[col])
    plt.title(f'Boxplot of {col}')
    plt.show()

# Correlation matrix for numeric columns
plt.figure(figsize=(12, 8))
sns.heatmap(train_data[numeric_cols].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()


      id  Gender  ...                 MTRANS           NObeyesdad
0   9958    Male  ...             Automobile       Obesity_Type_I
1   7841    Male  ...  Public_Transportation  Insufficient_Weight
2   9293    Male  ...  Public_Transportation      Obesity_Type_II
3  15209  Female  ...             Automobile       Obesity_Type_I
4  16515    Male  ...  Public_Transportation  Overweight_Level_II

[5 rows x 18 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16606 entries, 0 to 16605
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              16606 non-null  int64  
 1   Gender                          16606 non-null  object 
 2   Age                             16606 non-null  float64
 3   Height                          16606 non-null  float64
 4   Weight                          16606 non-null  float64
 5   family_history_with_overweight  16606 no

Numeric Columns: Index(['id', 'Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE'], dtype='object')
Categorical Columns: Index(['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE',
       'SCC', 'CALC', 'MTRANS', 'NObeyesdad'],
      dtype='object')
                 id           Age  ...           FAF           TUE
count  16606.000000  16606.000000  ...  16606.000000  16606.000000
mean   10363.210888     23.892845  ...      0.985767      0.619577
std     5999.344301      5.747219  ...      0.841048      0.604124
min        0.000000     14.000000  ...      0.000000      0.000000
25%     5171.250000     20.000000  ...      0.008822      0.000000
50%    10377.500000     22.832105  ...      1.000000      0.573958
75%    15542.500000     26.000000  ...      1.600536      1.000000
max    20757.000000     61.000000  ...      3.000000      2.000000

[8 rows x 9 columns]
        Gender  ...        NObeyesdad
count    16606  ...             16606
unique       2  ...   

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data)
print("column_info")
print(column_info)


2025-09-15 00:00:56.324 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS', 'NObeyesdad'], 'Numeric': ['id', 'Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, StandardScale

# Load the test data
test_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/obesity_CVD_risk/test.csv')

# Copy the data to avoid modifying the original data
train_data_copy = train_data.copy()
test_data_copy = test_data.copy()

# Handle missing values
numeric_cols = train_data_copy.select_dtypes(include=[np.number]).columns
categorical_cols = train_data_copy.select_dtypes(include=['object', 'category']).columns

# Fill missing values for numeric columns with mean
fill_missing_numeric = FillMissingValue(features=numeric_cols, strategy='mean')
train_data_copy = fill_missing_numeric.fit_transform(train_data_copy)
test_data_copy = fill_missing_numeric.transform(test_data_copy)

# Fill missing values for categorical columns with most frequent value
fill_missing_categorical = FillMissingValue(features=categorical_cols, strategy='most_frequent')
train_data_copy = fill_missing_categorical.fit_transform(train_data_copy)
test_data_copy = fill_missing_categorical.transform(test_data_copy)

# Encode categorical variables using label encoding
label_encode = LabelEncode(features=categorical_cols)
train_data_copy = label_encode.fit_transform(train_data_copy)
test_data_copy = label_encode.transform(test_data_copy)

# Normalize numerical features
standard_scale = StandardScale(features=numeric_cols)
train_data_copy = standard_scale.fit_transform(train_data_copy)
test_data_copy = standard_scale.transform(test_data_copy)

# Display the first few rows of the processed data
print(train_data_copy.head())
print(test_data_copy.head())


         id  Gender       Age    Height  ...       TUE  CALC  MTRANS  NObeyesdad
0 -0.067545       1 -1.199372  0.800816  ...  0.629728     2       0           2
1 -0.420427       1 -0.213196  0.610243  ...  0.629728     2       3           0
2 -0.178393       1 -0.357492  1.372928  ... -1.025610     1       3           3
3  0.807744       0  2.976687 -1.379006  ... -1.025610     1       0           2
4  1.025441       1 -0.155357  1.144999  ...  0.629728     0       3           6

[5 rows x 18 columns]
         id  Gender       Age    Height  ...       TUE  CALC  MTRANS  NObeyesdad
0 -0.007703       0  0.366650 -0.585344  ...  0.151883     1       3           4
1 -1.048348       1 -1.025370  0.571361  ...  0.629728     1       3           5
2 -0.217232       0 -0.092663  0.151229  ... -1.021306     2       3           2
3  0.153820       1  1.063013 -0.338806  ... -0.836145     1       3           3
4 -0.351584       1 -1.199372 -1.034823  ...  2.285067     2       3           0

[5 r

In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data_copy)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['id', 'Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', 'NObeyesdad'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from metagpt.tools.libs.data_preprocess import LabelEncode

# Assuming train_data_copy and test_data_copy are already preprocessed
X_train = train_data_copy.drop(columns=['NObeyesdad'])
y_train = train_data_copy['NObeyesdad']
X_test = test_data_copy.drop(columns=['NObeyesdad'])
y_test = test_data_copy['NObeyesdad']

# Initialize and train the RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))


Accuracy: 0.89
Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.92      0.93       524
           1       0.81      0.89      0.85       626
           2       0.90      0.87      0.88       543
           3       0.97      0.98      0.97       657
           4       1.00      1.00      1.00       804
           5       0.80      0.71      0.75       484
           6       0.78      0.82      0.80       514

    accuracy                           0.89      4152
   macro avg       0.89      0.88      0.88      4152
weighted avg       0.90      0.89      0.89      4152



In [6]:
# Load the test data
test_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/obesity_CVD_risk/test.csv')

# Apply the same preprocessing steps as the training data
test_data_copy = test_data.copy()
test_data_copy = fill_missing_numeric.transform(test_data_copy)
test_data_copy = fill_missing_categorical.transform(test_data_copy)
test_data_copy = label_encode.transform(test_data_copy)
test_data_copy = standard_scale.transform(test_data_copy)

# Prepare the features and target for the test set
X_test = test_data_copy.drop(columns=['NObeyesdad'])
y_test = test_data_copy['NObeyesdad']

# Predict using the trained model
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))


Accuracy: 0.89
Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.92      0.93       524
           1       0.81      0.89      0.85       626
           2       0.90      0.87      0.88       543
           3       0.97      0.98      0.97       657
           4       1.00      1.00      1.00       804
           5       0.80      0.71      0.75       484
           6       0.78      0.82      0.80       514

    accuracy                           0.89      4152
   macro avg       0.89      0.88      0.88      4152
weighted avg       0.90      0.89      0.89      4152

